In [2]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models

# 1. 加载数据 (碎片时间建议：先跑通数据下载)
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

#归一化：将像素值缩放到 0——1 之间
train_images , test_images = train_images / 255.0, test_images / 255.0

#定义一个数据增强层（像积木一样插在模型最前面）
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),  #随机左右翻转
    layers.RandomRotation(0.1),       #随机旋转一个小角度
    layers.RandomZoom(0.1),           #随机缩放
])

model = models.Sequential([
    # 第一层卷积：提取颜色，边缘特征
    # 32: 用 32 个卷积核，每个核提取一种基础特征（边缘、颜色、横线）；
    # (3,3): 卷积核大小，图像分类最通用的尺寸；
    # input_shape=(32,32,3): 输入图片尺寸（CIFAR10: 32×32 彩色图，3=RGB 三通道）。
    data_augmentation,
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.MaxPooling2D((2,2)),
    
    #第二层卷积：提取更复杂的纹理
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    
    #第三层卷积：提取物体零件
    layers.Conv2D(64, (3,3), activation='relu'),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    #Dropout层：随机让20%的神经元下班，防止过拟合
    layers.Dropout(0.2),
    layers.Dense(10) #最终输出10类
])

# 3.编译与训练
model.compile(optimizer='adam',
             loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
             metrics=['accuracy'])

model.fit(train_images, train_labels, epochs=5, validation_data=(test_images, test_labels))






Epoch 1/5


1563/1563 [==============================] - 16s 10ms/step - loss: 1.7429 - accuracy: 0.3591 - val_loss: 1.4057 - val_accuracy: 0.4899
Epoch 2/5
1563/1563 [==============================] - 13s 9ms/step - loss: 1.4570 - accuracy: 0.4755 - val_loss: 1.2746 - val_accuracy: 0.5438
Epoch 3/5
1563/1563 [==============================] - 14s 9ms/step - loss: 1.3480 - accuracy: 0.5186 - val_loss: 1.2224 - val_accuracy: 0.5617
Epoch 4/5
1563/1563 [==============================] - 15s 9ms/step - loss: 1.2807 - accuracy: 0.5456 - val_loss: 1.1536 - val_accuracy: 0.5897
Epoch 5/5
1563/1563 [==============================] - 14s 9ms/step - loss: 1.2356 - accuracy: 0.5606 - val_loss: 1.2216 - val_accuracy: 0.5735
